# Polars Streaming — Process Files Larger Than Memory

Use Polars `scan_*` (lazy evaluation) and `sink_*` (streaming writes) to
process datasets that exceed available RAM.

| Approach | Function | Memory Usage |
|---|---|---|
| Eager | `read_parquet` → `run()` | Proportional to **file size** |
| Streaming | `scan_parquet` → `run_source_streaming()` | Proportional to **chunk size** |

## 1. Generate Sample Data

In [ ]:
from pathlib import Path
import polars as pl
import time

OUTPUT = Path("data")
OUTPUT.mkdir(exist_ok=True)

n_rows = 100_000
sample = pl.DataFrame({
    "event_id": [f"EVT-{i:06d}" for i in range(n_rows)],
    "event_type": ["click" if i % 3 == 0 else "view" if i % 3 == 1 else "purchase"
                   for i in range(n_rows)],
    "user_id": [f"user_{i % 1000:04d}" for i in range(n_rows)],
    "timestamp": [f"2024-12-{(i % 28) + 1:02d}T{(i % 24):02d}:00:00" for i in range(n_rows)],
    "value": [float(i % 500) - 10.0 for i in range(n_rows)],  # Some negatives
    "metadata": [f"source=web,session={i}" for i in range(n_rows)],
})

source_file = OUTPUT / "events.parquet"
sample.write_parquet(str(source_file))
file_size_mb = source_file.stat().st_size / (1024 * 1024)

print(f"Created: {source_file}")
print(f"Rows:    {n_rows:,}")
print(f"Size:    {file_size_mb:.1f} MB")
sample.head(5)

## 2. Eager Loading (Traditional)

Loads the **entire file into memory** before processing.

In [ ]:
from lakelogic.core.processor import DataProcessor

proc = DataProcessor(str(Path("streaming_contract.yaml")), engine="polars")

t0 = time.perf_counter()
df_eager = pl.read_parquet(str(source_file))
result_eager = proc.run(df_eager)
t1 = time.perf_counter()

good_eager = result_eager.good
if isinstance(good_eager, pl.LazyFrame):
    good_eager = good_eager.collect()
bad_eager = result_eager.bad
if isinstance(bad_eager, pl.LazyFrame):
    bad_eager = bad_eager.collect()

print(f"Time:      {t1 - t0:.3f}s")
print(f"Good rows: {good_eager.height:,}")
print(f"Bad rows:  {bad_eager.height:,}")
print(f"Approach:  Entire file loaded into memory first")

## 3. Streaming (Lazy `scan_parquet`)

Uses `scan_parquet()` under the hood — data is **not loaded until needed**.

In [ ]:
t0 = time.perf_counter()
result_stream = proc.run_source_streaming(str(source_file))
t1 = time.perf_counter()

good_stream = result_stream.good
if isinstance(good_stream, pl.LazyFrame):
    good_stream = good_stream.collect()
bad_stream = result_stream.bad
if isinstance(bad_stream, pl.LazyFrame):
    bad_stream = bad_stream.collect()

print(f"Time:      {t1 - t0:.3f}s")
print(f"Good rows: {good_stream.height:,}")
print(f"Bad rows:  {bad_stream.height:,}")
print(f"Approach:  Polars scans and streams in chunks")

## 4. Streaming with Sink

Write validated data **directly to disk** without ever calling `.collect()`.
This means the full dataset is **never in memory at once**.

In [ ]:
output_file = OUTPUT / "validated_events.parquet"

t0 = time.perf_counter()
result_sink = proc.run_source_streaming(str(source_file), output_path=str(output_file))
t1 = time.perf_counter()

output_size_mb = output_file.stat().st_size / (1024 * 1024)
readback = pl.read_parquet(str(output_file))

print(f"Time:        {t1 - t0:.3f}s")
print(f"Output:      {output_file}")
print(f"Output size: {output_size_mb:.1f} MB")
print(f"Output rows: {readback.height:,}")
print(f"Approach:    Sink directly to disk, constant memory")

## 5. Verify Results Match

In [ ]:
print(f"Eager good rows:     {good_eager.height:,}")
print(f"Streaming good rows: {good_stream.height:,}")
print(f"Sink output rows:    {readback.height:,}")
print(f"Results match:       {good_eager.height == good_stream.height}")

## 6. Streaming CSV

In [ ]:
csv_file = OUTPUT / "events.csv"
sample.write_csv(str(csv_file))

result_csv = proc.run_source_streaming(str(csv_file))

good_csv = result_csv.good
if isinstance(good_csv, pl.LazyFrame):
    good_csv = good_csv.collect()

print(f"CSV streaming good rows: {good_csv.height:,}")

## Summary

| Mode | Memory | When to Use |
|---|---|---|
| `proc.run(df)` | Full dataset | Small/medium files |
| `proc.run_source_streaming(path)` | Chunk-sized | Large files |
| `proc.run_source_streaming(path, output_path=...)` | Constant | ETL pipelines |

**Key advantage**: A 50GB file can be processed on an 8GB machine using streaming.